# MT-Bench Category Analysis (Likert breakdown + charts)

In [9]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [ ]:
MT_BENCH_DIR = Path("data/mt_bench")

QUESTION_PATH = Path("/Users/valeriosantini/Desktop/Final_AML/data/mt_bench/question.jsonl")
JUDGMENT_PATH = Path("/Users/valeriosantini/Desktop/Final_AML/data/mt_bench/model_judgment/gpt-4_single.jsonl")
PER_QUESTION_CSV = Path("/Users/valeriosantini/Desktop/Final_AML/MT-Bench/mtbench_results_out/per_question_scores.csv")

print("QUESTION:", QUESTION_PATH.resolve())
print("JUDGMENT :", JUDGMENT_PATH.resolve())
print("PER_Q    :", PER_QUESTION_CSV.resolve())


QUESTION: /Users/valeriosantini/Desktop/Final_AML/data/mt_bench/question.jsonl
JUDGMENT : /Users/valeriosantini/Desktop/Final_AML/data/mt_bench/model_judgment/gpt-4_single.jsonl
PER_Q    : /Users/valeriosantini/Desktop/Final_AML/MT-Bench/mtbench_results_out/per_question_scores.csv


In [ ]:
q_rows = []
with QUESTION_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        o = json.loads(line)
        q_rows.append({"question_id": o["question_id"], "category": o.get("category")})
qdf = pd.DataFrame(q_rows)
display(qdf.head())
print("Categories:", sorted(qdf["category"].unique().tolist()))


,question_id,category
0,81,writing
1,82,writing
2,83,writing
3,84,writing
4,85,writing


Categories: ['coding', 'extraction', 'humanities', 'math', 'reasoning', 'roleplay', 'stem', 'writing']


In [ ]:
def load_per_question_from_judgments(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            o = json.loads(line)
            rows.append({
                "model": o.get("model") or o.get("model_name"),
                "question_id": o.get("question_id"),
                "turn": o.get("turn"),
                "score": float(o.get("score")),
            })
    j = pd.DataFrame(rows)
    j = j.sort_index().drop_duplicates(subset=["model","question_id","turn"], keep="last")
    per_q = j.groupby(["model","question_id"])["score"].mean().reset_index()
    return per_q

if JUDGMENT_PATH.exists():
    per_q = load_per_question_from_judgments(JUDGMENT_PATH)
    print("Loaded per-question scores from judgments.")
elif PER_QUESTION_CSV.exists():
    per_q = pd.read_csv(PER_QUESTION_CSV)
    print("Loaded per-question scores from CSV.")
else:
    raise FileNotFoundError("Provide either judgments file or per_question_scores.csv")

display(per_q.head())
print("Rows:", len(per_q), "Models:", sorted(per_q["model"].unique().tolist()))


Loaded per-question scores from judgments.


,model,question_id,score
0,GrokLlama,81,4.5
1,GrokLlama,82,9.0
2,GrokLlama,83,6.5
3,GrokLlama,84,8.5
4,GrokLlama,85,10.0


Rows: 240 Models: ['GrokLlama', 'Llama2', 'WildLlama']


In [ ]:
df = per_q.merge(qdf, on="question_id", how="left")
assert df["category"].isna().sum() == 0

cat_order = ["coding","extraction","humanities","math","reasoning","roleplay","stem","writing"]
radar_order = ["humanities","extraction","coding","writing","stem","roleplay","reasoning","math"]

cat_summary = (
    df.groupby(["model","category"])["score"]
      .agg(count="count", mean="mean", median="median", std="std")
      .reset_index()
)
cat_summary["sem"] = cat_summary["std"] / np.sqrt(cat_summary["count"].clip(lower=1))

pivot_mean = cat_summary.pivot(index="category", columns="model", values="mean").reindex(cat_order)
display(cat_summary.sort_values(["category","mean"], ascending=[True,False]))
display(pivot_mean)


,model,category,count,mean,median,std,sem
8,Llama2,coding,10,3.000,1.750,2.728451,0.862812
16,WildLlama,coding,10,2.750,1.750,2.530371,0.800174
0,GrokLlama,coding,10,1.750,2.000,0.485913,0.153659
9,Llama2,extraction,10,6.500,6.750,2.677063,0.846562
17,WildLlama,extraction,10,4.900,4.750,2.157674,0.682316
1,GrokLlama,extraction,10,3.550,3.500,2.385721,0.754431
18,WildLlama,humanities,10,9.775,10.000,0.299305,0.094648
10,Llama2,humanities,10,8.750,9.500,1.918477,0.606676
2,GrokLlama,humanities,10,8.500,8.500,1.247219,0.394405
11,Llama2,math,10,2.400,1.750,2.705960,0.855700


model,GrokLlama,Llama2,WildLlama
category,,,
coding,1.75,3.00,2.750
extraction,3.55,6.50,4.900
humanities,8.50,8.75,9.775
math,1.25,2.40,2.400
reasoning,3.80,4.25,4.150
roleplay,6.35,7.70,7.725
stem,7.95,8.65,8.050
writing,7.55,8.90,8.550


In [ ]:
def radar_plot(
    pivot_means: pd.DataFrame,
    categories: list[str],
    title: str,
    save_path: Path,
    fill=False,
):
    model_colors = {
        "GrokLlama": "#20aa59",
        "Llama2": "#3490db",
        "WildLlama": "#d12c2c",
    }

    cats = categories[:]
    N = len(cats)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure()
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    ax.set_thetagrids(
        np.degrees(angles[:-1]),
        [c.capitalize() for c in cats],
    )
    ax.set_ylim(0, 10)
    ax.set_yticks([0, 2, 4, 6, 8, 10])
    ax.set_yticklabels(["0", "2", "4", "6", "8", "10"])

    for model in pivot_means.columns:
        vals = pivot_means.loc[cats, model].values.tolist()
        vals += vals[:1]

        color = model_colors.get(model, None)

        ax.plot(
            angles,
            vals,
            marker="o",
            label=model,
            color=color,
        )

        if fill:
            ax.fill(
                angles,
                vals,
                color=color,
                alpha=0.08,
            )

    ax.set_title(title, pad=18)
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.15))
    plt.tight_layout()
    fig.savefig(save_path, dpi=200)
    plt.close(fig)


In [ ]:
cat_summary.to_csv(out_dir / "mtbench_category_summary_long.csv", index=False)
pivot_mean.to_csv(out_dir / "mtbench_category_means_pivot.csv")

overall = df.groupby("model")["score"].agg(count="count", mean="mean", median="median", std="std")
overall["sem"] = overall["std"] / np.sqrt(overall["count"].clip(lower=1))
overall.reset_index().to_csv(out_dir / "mtbench_overall_from_per_question.csv", index=False)

print("Saved CSVs in:", out_dir.resolve())


Saved CSVs in: /Users/valeriosantini/Desktop/Final_AML/MT-Bench/mtbench_category_outputs


In [ ]:
pivot_radar = cat_summary.pivot(index="category", columns="model", values="mean").reindex(radar_order)

radar_plot(pivot_radar, radar_order, "MT-Bench Likert Breakdown by Category (mean)", out_dir / "radar_compare.png")

for m in pivot_radar.columns:
    radar_plot(pivot_radar[[m]], radar_order, f"MT-Bench Likert by Category — {m}", out_dir / f"radar_{m}.png", fill=True)

bar_df = pivot_mean.reset_index()
x = np.arange(len(bar_df["category"]))
fig = plt.figure()
ax = plt.gca()
width = 0.25 if len(pivot_mean.columns) >= 3 else 0.35
models = list(pivot_mean.columns)

for i, m in enumerate(models):
    ax.bar(x + (i - (len(models)-1)/2)*width, bar_df[m].values, width, label=m)

ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in bar_df["category"]], rotation=20, ha="right")
ax.set_ylabel("Mean Likert score (1–10)")
ax.set_title("MT-Bench mean score by category")
ax.set_ylim(0, 10)
ax.legend()
plt.tight_layout()
fig.savefig(out_dir / "bar_category_compare.png", dpi=200)
plt.close(fig)

print("Saved PNGs in:", out_dir.resolve())


Saved PNGs in: /Users/valeriosantini/Desktop/Final_AML/MT-Bench/mtbench_category_outputs
